# OpenPlaque — RCA Source-Space Plaque Quantification v1

Research-only source-CCTA experiment. This notebook runs end-to-end with **Runtime → Run all**.

It quantifies physical mm³ plaque-composition **proxy** volumes on the frozen accepted RCA and explicitly does **not** claim validated clinical TPV.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, sys

DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'RCA_Source_Space_Plaque_Quantification_v1'
REUSE_VALID_CACHE = False
print('Drive root:', DRIVE_ROOT)
print('Output:', OUTPUT)
print('Reuse valid cache:', REUSE_VALID_CACHE)


In [ ]:
# Fresh deterministic repository setup.
os.chdir('/content')
REPO = Path('/content/OpenPlaque_rca_plaque_v1')
if REPO.exists():
    shutil.rmtree(REPO)

!git clone -q --branch rca-source-space-plaque-quantification-from-main https://github.com/pazzani/OpenPlaque.git {REPO}
%cd {REPO}
!git checkout -q d6a13a20ee34e9fec47fe51c56775525183f4710

HEAD = !git rev-parse HEAD
MB = !git merge-base HEAD 0593b453959f5a353d644267fbeef24b514ef4d7
print('Checked out:', HEAD[0])
print('Merge base:', MB[0])
assert HEAD[0] == 'd6a13a20ee34e9fec47fe51c56775525183f4710'
assert MB[0] == '0593b453959f5a353d644267fbeef24b514ef4d7'

%pip install -q .
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
print('Installed pinned science commit.')


In [ ]:
# Synthetic test + targeted pytest in the notebook kernel.
from openplaque.rca_source_space_plaque_quantification_v1 import synthetic_self_test
print('Synthetic:', synthetic_self_test())

!python -m pytest -q tests/test_rca_source_space_plaque_quantification_v1.py


In [ ]:
# Preflight required Drive inputs before science.
required = [
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    DRIVE_ROOT/'Cache/Source_Volume_Coronary_Centerlines/RCA_source_centerline.csv',
    DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
    DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/accepted_anatomy.csv',
]
missing=[str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required inputs:\n'+'\n'.join(missing))
print('Preflight complete.')
print('Prior plaque profile available:', (DRIVE_ROOT/'Longitudinal_Plaque_PCAT_Fusion_v1/RCA_source_longitudinal_plaque_profile_1mm.csv').exists())
print('Locked PCAT profile available:', (DRIVE_ROOT/'PCAT_RCA_10_50_Reproducibility_Lock/pcat_canonical_primary_longitudinal.csv').exists())


In [ ]:
# Clean incomplete output unless a valid completed result is explicitly being reused.
if OUTPUT.exists() and not REUSE_VALID_CACHE:
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'notebook_started.json').write_text(json.dumps({'status':'NOTEBOOK_STARTED'}))
print('Output prepared:', OUTPUT)


In [ ]:
from openplaque.rca_source_space_plaque_quantification_v1 import run
result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
print('Report:', result['report'])
print('ZIP:', result['zip'])


In [ ]:
# Final artifact check.
expected = [
    'run_state.json','summary.json','input_provenance.json',
    'RCA_source_space_station_quantification.csv',
    'RCA_source_space_plaque_profile_1mm.csv',
    'RCA_shell_sensitivity.csv',
    '01_RCA_source_space_plaque_profile.png',
    '02_RCA_orthogonal_source_QC.png',
    '03_RCA_shell_sensitivity.png',
    'OPENPLAQUE_RCA_SOURCE_SPACE_PLAQUE_QUANTIFICATION_REPORT.html',
    'OPENPLAQUE_RCA_SOURCE_SPACE_PLAQUE_QUANTIFICATION_RESULTS.zip',
]
for name in expected:
    p=OUTPUT/name
    print(('OK  ' if p.exists() else 'MISS'), name)
assert all((OUTPUT/name).exists() for name in expected)
print('COMPLETE')
